# Lab 13: Implementing iLQG on the Hill-Type Muscle Plant

**Course:** Computational Sensorimotor Control | Week 13 of 15

**Task:** A 12 cm rightward reach from the reference posture (Q_REF: shoulder 55°, elbow 75°). The hand starts at the resting position and reaches 12 cm to the right along the x-axis. Movement duration is 500 ms. The control interval is 5 ms (100 control steps), with each step internally simulated at 1 ms (5 substeps). The plant is the 6-muscle Hill-type arm from Week 11 with a 16-dimensional state vector.

**Prerequisites:** Working `riccati_backward()` from Lab 12. The `smc` package (v0.2.0) with the `plant16d` module.


## Setup

In [ ]:
# Install the smc package (run once)
!pip install --break-system-packages -q git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
from smc import (
    Arm, Q_REF, make_hill_muscles,
    simulate_hill, lambda_for_posture, make_ramp,
    NX, NU, MUSCLE_NAMES, DT_SIM, DT_CTRL, N_SUBSTEPS,
    make_x0, make_target, set_muscle_state, hill_step, forward_rollout,
)

arm = Arm()
start_hand = arm.forward_kinematics(Q_REF)
target_hand = start_hand + np.array([0.12, 0.0])
q_target = arm.inverse_kinematics(*target_hand)

T = 0.5; N = int(T / DT_CTRL)
print(f"NX={NX}, NU={NU}, N={N}, dt_ctrl={DT_CTRL*1000:.0f} ms")


## Part 1: Verify the 16D Forward Model

The 16D state: $\mathbf{x} = [\theta_1, \theta_2, \dot{\theta}_1, \dot{\theta}_2, \gamma_1 \ldots \gamma_6, l_{CE,1} \ldots l_{CE,6}]$

**No code to write.** Run and verify endpoints agree within ~0.5 cm.


In [ ]:
li = lambda_for_posture(Q_REF); lf = lambda_for_posture(q_target)
lam_fn = make_ramp(li, lf, 0.05, 0.25)
t_eph, _, hand_eph, stim_eph = simulate_hill(lam_fn, T=T, dt=DT_SIM)

u_eph = np.zeros((N, NU))
for t in range(N): u_eph[t] = np.mean(stim_eph[t*N_SUBSTEPS:(t+1)*N_SUBSTEPS], axis=0)

x0 = make_x0(Q_REF); xs = make_target(q_target)
x_traj_eph = forward_rollout(x0, u_eph)

hand_16d = arm.forward_kinematics(x_traj_eph[-1, :2])
print(f"simulate_hill:   ({hand_eph[-1,0]*100:.2f}, {hand_eph[-1,1]*100:.2f}) cm")
print(f"forward_rollout: ({hand_16d[0]*100:.2f}, {hand_16d[1]*100:.2f}) cm")
print(f"Difference: {np.linalg.norm(np.array(hand_16d)-hand_eph[-1])*100:.2f} cm")


## Part 2: Computing Jacobians via Finite Differences

$$A_t[:, j] = \frac{f(\bar{\mathbf{x}} + \varepsilon \mathbf{e}_j, \bar{\mathbf{u}}) - f(\bar{\mathbf{x}} - \varepsilon \mathbf{e}_j, \bar{\mathbf{u}})}{2\varepsilon}, \qquad B_t[:, j] = \frac{f(\bar{\mathbf{x}}, \bar{\mathbf{u}} + \varepsilon \mathbf{e}_j) - f(\bar{\mathbf{x}}, \bar{\mathbf{u}} - \varepsilon \mathbf{e}_j)}{2\varepsilon}$$

**Critical:** Each perturbation needs fresh muscles via `set_muscle_state()`.

### Exercise 2: Implement `compute_jacobians` (2 lines)


In [ ]:
def compute_jacobians(x_bar, u_bar, eps=1e-5):
    A = np.zeros((NX, NX)); B = np.zeros((NX, NU))
    for j in range(NX):
        xp = x_bar.copy(); xp[j] += eps
        xm = x_bar.copy(); xm[j] -= eps
        mp = make_hill_muscles(); set_muscle_state(mp, xp)
        mm = make_hill_muscles(); set_muscle_state(mm, xm)
        ### YOUR CODE: compute A[:, j] using hill_step ###
        # A[:, j] = ...
    for j in range(NU):
        up = u_bar.copy(); up[j] += eps
        um = u_bar.copy(); um[j] -= eps
        mp = make_hill_muscles(); set_muscle_state(mp, x_bar)
        mm = make_hill_muscles(); set_muscle_state(mm, x_bar)
        ### YOUR CODE: compute B[:, j] using hill_step ###
        # B[:, j] = ...
    return A, B

# Quick test
A0, B0 = compute_jacobians(x_traj_eph[0], u_eph[0])
print(f"A: {A0.shape}, B: {B0.shape}, A diag[:4]={np.diag(A0)[:4].round(3)}")


## Part 3: Extended Riccati Backward Pass

New recursion (added to Lab 12's $S_t$):

$$v_N = Q_N(\bar{\mathbf{x}}_N - \mathbf{x}^*), \qquad L_v = (B^\top S_{t+1} B + R)^{-1} B^\top, \qquad v_t = (A - BL)^\top v_{t+1} - L^\top R \bar{\mathbf{u}}_t + Q\bar{\mathbf{x}}_t$$

### Exercise 3: Add the v_t machinery (3 lines)


In [ ]:
def riccati_backward_ilqg(A_list, B_list, Q, QN, R, x_nom, u_nom, xs):
    N = len(A_list)
    S=[None]*(N+1); v=[None]*(N+1); L=[None]*N; Lv=[None]*N; Lu=[None]*N
    S[N] = QN.copy()
    ### YOUR CODE: v[N] = ? ###
    # v[N] = ...
    for t in range(N-1, -1, -1):
        A, B = A_list[t], B_list[t]
        H_inv = np.linalg.inv(B.T @ S[t+1] @ B + R)
        L[t] = H_inv @ B.T @ S[t+1] @ A
        ### YOUR CODE: Lv[t] = ? ###
        # Lv[t] = ...
        Lu[t] = H_inv @ R
        A_cl = A - B @ L[t]
        S[t] = A.T @ S[t+1] @ A_cl + Q
        ### YOUR CODE: v[t] = ? ###
        # v[t] = ...
    return L, Lv, Lu, S, v

# Cost matrices
QN = np.zeros((NX,NX)); QN[:4,:4] = np.diag([5000.,5000.,100.,100.])
Q = np.zeros((NX,NX)); R = 0.001*np.eye(NU)

# Linearise EPH trajectory
print("Linearising EPH trajectory...")
A_list, B_list = [], []
for t in range(N): A,B = compute_jacobians(x_traj_eph[t], u_eph[t]); A_list.append(A); B_list.append(B)
L, Lv, Lu, S, v = riccati_backward_ilqg(A_list, B_list, Q, QN, R, x_traj_eph, u_eph, xs)
print(f"v[N] matches QN(x̄_N−x*): {np.allclose(v[N], QN@(x_traj_eph[N]-xs))}")


## Part 4: Forward Pass with Three-Term Control Law

$$\delta \mathbf{u}_t = -L_t \delta \mathbf{x}_t - \alpha L_v v_{t+1} - \alpha L_u \bar{\mathbf{u}}_t$$

### Exercise 4: Implement the forward pass (2 lines)


In [ ]:
def forward_pass(x0, u_nom, x_nom, L, Lv, Lu, v, alpha=1.0):
    N = len(u_nom)
    x_new = np.zeros((N+1,NX)); u_new = np.zeros((N,NU)); x_new[0] = x0
    muscles = make_hill_muscles(); set_muscle_state(muscles, x0)
    for t in range(N):
        dx = x_new[t] - x_nom[t]
        ### YOUR CODE: compute du (three-term control law) ###
        # du = ...
        ### YOUR CODE: total control = nominal + correction, clipped ###
        # u_new[t] = ...
        x_new[t+1] = hill_step(x_new[t], u_new[t], muscles)
    return x_new, u_new

def compute_cost(x_traj, u_seq, QN, R, xs):
    N = len(u_seq); dx = x_traj[N]-xs
    return 0.5*dx@QN@dx + sum(0.5*u_seq[t]@R@u_seq[t] for t in range(N))

# Test
c0 = compute_cost(x_traj_eph, u_eph, QN, R, xs)
x_try, u_try = forward_pass(x0, u_eph, x_traj_eph, L, Lv, Lu, v, alpha=0.1)
c1 = compute_cost(x_try, u_try, QN, R, xs)
print(f"Cost: {c0:.4f} → {c1:.4f} ({'improved!' if c1<c0 else 'NOT improved'})")


## Part 5: The Full iLQG Loop

### Exercise 5: Complete the loop (3 lines)


In [ ]:
def run_ilqg(x0, xs, u_init, QN, Q, R, max_iter=8, tol=1e-6):
    N = len(u_init)
    u_curr = u_init.copy(); x_curr = forward_rollout(x0, u_curr)
    cost_curr = compute_cost(x_curr, u_curr, QN, R, xs)
    costs = [cost_curr]
    hand_trajs = [np.array([arm.forward_kinematics(x_curr[t,:2]) for t in range(N+1)])]
    print(f"  Init: cost={cost_curr:.4f}, err={np.linalg.norm(arm.forward_kinematics(x_curr[N,:2])-target_hand)*100:.3f} cm")
    for it in range(max_iter):
        A_list, B_list = [], []
        for t in range(N): A,B = compute_jacobians(x_curr[t], u_curr[t]); A_list.append(A); B_list.append(B)
        L, Lv, Lu, S, v = riccati_backward_ilqg(A_list, B_list, Q, QN, R, x_curr, u_curr, xs)
        improved = False
        for alpha in [1.0, 0.5, 0.25, 0.1, 0.05, 0.01]:
            x_try, u_try = forward_pass(x0, u_curr, x_curr, L, Lv, Lu, v, alpha)
            cost_try = compute_cost(x_try, u_try, QN, R, xs)
            if cost_try < cost_curr:
                ### YOUR CODE: update x_curr, u_curr, cost_curr ###
                # x_curr = ...
                # u_curr = ...
                # cost_curr = ...
                improved = True; break
        costs.append(cost_curr)
        hand_trajs.append(np.array([arm.forward_kinematics(x_curr[t,:2]) for t in range(N+1)]))
        h = arm.forward_kinematics(x_curr[N,:2])
        print(f"  It {it+1}: cost={cost_curr:.4f}, err={np.linalg.norm(np.array(h)-target_hand)*100:.3f} cm, α={'%.2f'%alpha if improved else 'none'}")
        ### YOUR CODE: check convergence ###
        # if ...
        #     break
    return x_curr, u_curr, costs, hand_trajs

print("Running iLQG...")
x_ilqg, u_ilqg, costs, hand_trajs = run_ilqg(x0, xs, u_eph, QN, Q, R)


## Reproduce Figure 4: iLQG Convergence

The lecture's Figure 4 showed convergence on the torque-driven arm. Here you reproduce the equivalent for the Hill-type muscle plant: cost vs iteration (Panel E) and hand trajectories at each iteration (Panel A).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel A: Hand trajectories at each iteration
ax = axes[0]
colors_iter = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(hand_trajs)))
for i, (ht, c) in enumerate(zip(hand_trajs, colors_iter)):
    label = f'Iter {i}' + (' (EPH init)' if i==0 else ' (converged)' if i==len(hand_trajs)-1 else '')
    ls = '-' if i==0 or i==len(hand_trajs)-1 else '--'
    lw = 2.5 if i==0 or i==len(hand_trajs)-1 else 1.5
    ax.plot(ht[:,0]*100, ht[:,1]*100, color=c, lw=lw, ls=ls, label=label)
ax.scatter([start_hand[0]*100],[start_hand[1]*100], c='navy', s=60, zorder=10)
ax.scatter([target_hand[0]*100],[target_hand[1]*100], c='k', s=100, marker='*', zorder=10)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('A) Hand trajectories across iterations\n(cf. lecture Figure 4A)', fontweight='bold')
ax.legend(fontsize=8); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

# Panel B: Cost vs iteration
ax = axes[1]
ax.semilogy(range(len(costs)), costs, 'o-', color='#F39C12', lw=2.5, markersize=8)
ax.set_xlabel('Iteration'); ax.set_ylabel('Cost (log scale)')
ax.set_title('B) Cost vs iteration\n(cf. lecture Figure 4E)', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Reproduce Figure 5: How the Linearization Changes Along the Trajectory

Compute Jacobians along the **converged** iLQG trajectory and plot how each muscle's mechanical effect varies over time.


In [ ]:
print("Computing Jacobians along converged trajectory...")
B_along = np.zeros((N, NX, NU))
for t in range(N):
    _, B_along[t] = compute_jacobians(x_ilqg[t], u_ilqg[t])
print("Done.")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
t_ms = np.arange(N) * DT_CTRL * 1000
muscle_colors = ['#E74C3C','#27AE60','#2ECC71','#3498DB','#E67E22','#8E44AD']

# Panel A: ||B_t column|| for each muscle
ax = axes[0]
for j in range(NU):
    effect = np.sqrt(B_along[:,2,j]**2 + B_along[:,3,j]**2)
    ax.plot(t_ms, uniform_filter1d(effect, 15), color=muscle_colors[j], lw=2, label=MUSCLE_NAMES[j])
ax.set_xlabel('Time (ms)'); ax.set_ylabel('||B_t column||')
ax.set_title('A) Mechanical effect per muscle\n(cf. lecture Figure 5A)', fontweight='bold')
ax.legend(fontsize=7.5); ax.grid(True, alpha=0.3)

# Panel B: Direction pec and tri_lat push the hand
ax = axes[1]
for idx, name, color in [(0,'Pec','#E74C3C'), (4,'Tri_l','#E67E22')]:
    angles = []
    for t in range(N):
        J = arm.jacobian(x_ilqg[t,:2])
        b_hand = J @ np.array([B_along[t,2,idx], B_along[t,3,idx]])
        angles.append(np.degrees(np.arctan2(b_hand[1], b_hand[0])))
    ax.plot(t_ms, uniform_filter1d(np.array(angles),15), color=color, lw=2.5, label=name)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Hand push direction (°)')
ax.set_title('B) Push direction rotates\n(cf. lecture Figure 5B)', fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# Panel C: Key state variables
ax = axes[2]
t_state = np.arange(N+1)*DT_CTRL*1000
ax2 = ax.twinx()
ln1 = ax.plot(t_state, np.degrees(x_ilqg[:,0]-x_ilqg[0,0]), 'navy', lw=2, label='Δθ₁ (°)')
ln2 = ax.plot(t_state, x_ilqg[:,4]*50, '#E74C3C', lw=2, ls='--', label='γ_pec (×50)')
from smc.hill_muscle import HILL_MUSCLE_DEFS, HillMuscle
pec_m = HillMuscle(*HILL_MUSCLE_DEFS[0])
ln3 = ax2.plot(t_state, x_ilqg[:,10]/pec_m.l_CE_opt, '#F39C12', lw=2, ls=':', label='l_CE/l_opt')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Δθ₁ / γ (×50)')
ax2.set_ylabel('l_CE / l_opt', color='#F39C12')
lines = ln1+ln2+ln3; ax.legend(lines, [l.get_label() for l in lines], fontsize=8, loc='upper left')
ax.set_title('C) 16D state evolves\n(cf. lecture Figure 5C)', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Print variation summary
for j in range(NU):
    effect = np.sqrt(B_along[:,2,j]**2 + B_along[:,3,j]**2)
    es = uniform_filter1d(effect, 15)
    ratio = es.max()/max(es.min(),1e-10)
    print(f"  {MUSCLE_NAMES[j]:6s}: B_t varies {ratio:.0f}×")


## Part 6: Reproduce Figure 6 — EPH vs Single-Point vs Full iLQG

### Exercise 6: Single-point linearization (1 line)


In [ ]:
mid = N // 2
### YOUR CODE: compute fixed A, B at midpoint ###
# A_fixed, B_fixed = ...

u_sp = u_eph.copy(); x_sp = forward_rollout(x0, u_sp)
for it in range(8):
    L_sp, Lv_sp, Lu_sp, S_sp, v_sp = riccati_backward_ilqg(
        [A_fixed]*N, [B_fixed]*N, Q, QN, R, x_sp, u_sp, xs)
    cost_sp = compute_cost(x_sp, u_sp, QN, R, xs); improved=False
    for alpha in [1.0,0.5,0.25,0.1,0.05,0.01,0.005,0.001]:
        x_try, u_try = forward_pass(x0, u_sp, x_sp, L_sp, Lv_sp, Lu_sp, v_sp, alpha)
        if compute_cost(x_try, u_try, QN, R, xs) < cost_sp:
            x_sp,u_sp = x_try.copy(),u_try.copy()
            cost_sp = compute_cost(x_sp,u_sp,QN,R,xs); improved=True; break
    err = np.linalg.norm(arm.forward_kinematics(x_sp[N,:2])-target_hand)*100
    print(f"  SP it {it+1}: err={err:.3f} cm, α={'%.3f'%alpha if improved else 'none'}")
    if not improved: break

# Compare
errs = [np.linalg.norm(arm.forward_kinematics(x_traj_eph[-1,:2])-target_hand)*100,
        np.linalg.norm(arm.forward_kinematics(x_sp[-1,:2])-target_hand)*100,
        np.linalg.norm(arm.forward_kinematics(x_ilqg[-1,:2])-target_hand)*100]

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(['EPH\n(no lin.)', 'Single-point\n(midpoint)', 'iLQG\n(re-linearized)'],
              errs, color=['#E74C3C','#8E44AD','#F39C12'], edgecolor='black', linewidth=0.5)
for bar, err in zip(bars, errs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.04,
            f'{err:.2f} cm', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Endpoint error (cm)')
ax.set_title('Figure 6 (reproduced): Single-point helps, re-linearization helps more', fontweight='bold')
ax.set_ylim(0, max(errs)*1.4); plt.tight_layout(); plt.show()


## Part 7: State Vector Experiment — 10D vs 16D

### Exercise 7: Extract 10D state (1 line)


In [ ]:
def compute_jacobians_10d(x10, u_bar, eps=1e-5):
    nx10 = 10; A = np.zeros((nx10,nx10)); B = np.zeros((nx10,NU))
    for j in range(nx10):
        xp=x10.copy(); xp[j]+=eps; xm=x10.copy(); xm[j]-=eps
        xp16=np.zeros(NX); xp16[:10]=xp; xm16=np.zeros(NX); xm16[:10]=xm
        for k in range(NU):
            m=HillMuscle(*HILL_MUSCLE_DEFS[k])
            xp16[10+k]=m.mtc_length(xp[:2])-m.l_SE_0
            xm16[10+k]=m.mtc_length(xm[:2])-m.l_SE_0
        mp=make_hill_muscles(); set_muscle_state(mp,xp16)
        mm=make_hill_muscles(); set_muscle_state(mm,xm16)
        A[:,j]=(hill_step(xp16,u_bar,mp)[:10]-hill_step(xm16,u_bar,mm)[:10])/(2*eps)
    for j in range(NU):
        up=u_bar.copy(); up[j]+=eps; um=u_bar.copy(); um[j]-=eps
        x16=np.zeros(NX); x16[:10]=x10
        for k in range(NU):
            m=HillMuscle(*HILL_MUSCLE_DEFS[k]); x16[10+k]=m.mtc_length(x10[:2])-m.l_SE_0
        mp=make_hill_muscles(); set_muscle_state(mp,x16)
        mm=make_hill_muscles(); set_muscle_state(mm,x16)
        B[:,j]=(hill_step(x16,up,mp)[:10]-hill_step(x16,um,mm)[:10])/(2*eps)
    return A, B

### YOUR CODE: extract 10D state (drop l_CE) ###
# x0_10 = ...
# xs_10 = ...

nx10=10; QN_10=np.zeros((nx10,nx10)); QN_10[:4,:4]=np.diag([5000.,5000.,100.,100.])
Q_10=np.zeros((nx10,nx10)); R_10=0.001*np.eye(NU)
u_10=u_eph.copy(); x_10f=forward_rollout(x0,u_10); x_10=x_10f[:,:10]

print("Running 10D iLQG...")
for it in range(8):
    Al,Bl=[],[]
    for t in range(N): A,B=compute_jacobians_10d(x_10[t],u_10[t]); Al.append(A); Bl.append(B)
    S10=[None]*(N+1);v10=[None]*(N+1);L10=[None]*N;Lv10=[None]*N;Lu10=[None]*N
    S10[N]=QN_10.copy(); v10[N]=QN_10@(x_10[N]-xs_10)
    for t in range(N-1,-1,-1):
        A,B=Al[t],Bl[t]; Hi=np.linalg.inv(B.T@S10[t+1]@B+R_10)
        L10[t]=Hi@B.T@S10[t+1]@A; Lv10[t]=Hi@B.T; Lu10[t]=Hi@R_10
        Acl=A-B@L10[t]; S10[t]=A.T@S10[t+1]@Acl+Q_10
        v10[t]=Acl.T@v10[t+1]-L10[t].T@R_10@u_10[t]+Q_10@x_10[t]
    improved=False
    for alpha in [1.0,0.5,0.25,0.1,0.05,0.01,0.005,0.001]:
        muscles=make_hill_muscles();set_muscle_state(muscles,x0)
        un=np.zeros((N,NU));xn=np.zeros((N+1,NX));xn[0]=x0;ok=True
        for t in range(N):
            dx=xn[t,:10]-x_10[t];du=-L10[t]@dx-alpha*Lv10[t]@v10[t+1]-alpha*Lu10[t]@u_10[t]
            un[t]=np.clip(u_10[t]+du,0,1);xn[t+1]=hill_step(xn[t],un[t],muscles)
            if np.any(np.abs(xn[t+1,:4])>10) or np.any(np.isnan(xn[t+1])): ok=False; break
        if ok and compute_cost(xn,un,QN,R,xs)<compute_cost(x_10f,u_10,QN,R,xs):
            x_10f=xn.copy();u_10=un.copy();x_10=x_10f[:,:10];improved=True; break
    err=np.linalg.norm(arm.forward_kinematics(x_10f[N,:2])-target_hand)*100
    print(f"  It {it+1}: err={err:.3f} cm, α={'%.3f'%alpha if improved else 'none'}")
    if not improved: break

err_10=np.linalg.norm(arm.forward_kinematics(x_10f[-1,:2])-target_hand)*100
print(f"\n10D: {err_10:.2f} cm, EPH: {errs[0]:.2f} cm, 16D: {errs[2]:.2f} cm")
if err_10 > errs[0]: print("10D made things WORSE — incomplete linearization is harmful!")


## Part 8: Sensory Delays and the Augmented State (Stretch)

We continue with the same 12 cm rightward reach (500 ms, Q_REF start, 16D Hill plant). The iLQG controller from Part 5 assumed perfect state feedback. In reality, sensory feedback is **delayed**: proprioception arrives 40 ms late, vision arrives 100 ms late.

The proper treatment uses an **augmented state** that buffers past states so the Kalman filter can assign each observation to the correct time:

- **Naive KF (16D, wrong):** Observation $\mathbf{y}$ arrives at time $t$ but was measured at $t-8$. The naive KF applies the correction to $\hat{\mathbf{x}}(t)$, pulling the estimate toward where the arm was 40 ms ago.
- **Augmented KF (336D, correct):** Same observation, but $C_{\text{prop}}$ reads from the $t-8$ slot. The correction is applied to the right time. The buffer propagates it forward to improve $\hat{\mathbf{x}}(t)$.

**Same Kalman equations, same observation.** The only difference is that $C_{\text{prop}}$ points to the correct delayed slot.

### Exercise 8a: Build the augmented state matrices (3 lines)

$$A_{\text{aug}} = \begin{bmatrix} A_t & 0 & \cdots & 0 \\ I & 0 & \cdots & 0 \\ 0 & I & \cdots & 0 \\ \vdots & & \ddots & \vdots \\ 0 & \cdots & I & 0 \end{bmatrix} \quad (16 \times 21 = 336), \qquad C_{\text{prop}} = [0 \ldots 0 \; I_{4} \; 0 \ldots 0] \quad \text{(reads slot } t-8\text{)}$$


In [ ]:
PROP_DELAY = 8   # 40 ms / 5 ms
VIS_DELAY = 20   # 100 ms / 5 ms
D = VIS_DELAY
N_AUG = NX * (D + 1)
print(f"Augmented state: {NX} × {D+1} = {N_AUG} dimensions")

def build_A_aug(A_t):
    A_aug = np.zeros((N_AUG, N_AUG))
    ### YOUR CODE: place A_t in top-left block ###
    # A_aug[...] = ...
    ### YOUR CODE: identity shift blocks on sub-diagonal ###
    # for k in range(D):
    #     A_aug[...] = np.eye(NX)
    return A_aug

def build_C_prop():
    C = np.zeros((4, N_AUG))
    offset = PROP_DELAY * NX
    ### YOUR CODE: C reads first 4 state vars from delayed slot ###
    # C[...] = ...
    return C

def build_C_vis(q):
    J = arm.jacobian(q)
    C = np.zeros((4, N_AUG))
    offset = VIS_DELAY * NX
    C[0:2, offset:offset+2] = J
    C[2:4, offset+2:offset+4] = J
    return C

# Verify
A_aug = build_A_aug(A_conv[0])
C_prop = build_C_prop()
C_vis = build_C_vis(Q_REF)
print(f"A_aug: {A_aug.shape}, nonzero: {np.count_nonzero(A_aug)}/{N_AUG**2} ({np.count_nonzero(A_aug)/N_AUG**2*100:.1f}%)")
print(f"C_prop reads columns {np.where(C_prop.any(axis=0))[0]} (slot t−{PROP_DELAY})")
print(f"C_vis reads columns {np.where(C_vis.any(axis=0))[0]} (slot t−{VIS_DELAY})")
print(f"\nA_aug is {N_AUG}×{N_AUG} = {N_AUG**2:,} entries, but only "
      f"{np.count_nonzero(A_aug):,} are nonzero — the structure is almost all zeros and identities.")


### Exercise 8b: How Stale Are the Observations? (Reproduces Figure 7)

**No code to write.** The cell below computes how much information is lost to delay along the converged iLQG trajectory. During peak velocity, proprioception sees the shoulder angle from 40 ms ago and vision sees the hand position from 100 ms ago. The gap between the true and delayed signal is the **staleness** — the information the controller is missing.


In [ ]:
# Compute staleness along the converged trajectory
t_ctrl = np.arange(N+1) * DT_CTRL * 1000  # ms
hand_ilqg = np.array([arm.forward_kinematics(x_ilqg[t,:2]) for t in range(N+1)])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Panel A: Shoulder angle — true vs proprioception (40ms delay)
ax = axes[0]
theta1_true = np.degrees(x_ilqg[:, 0])
prop_delay_steps = int(0.040 / DT_CTRL)  # 8 steps
theta1_delayed = np.zeros(N+1)
for t in range(N+1):
    t_obs = max(0, t - prop_delay_steps)
    theta1_delayed[t] = theta1_true[t_obs]
ax.plot(t_ctrl, theta1_true, 'k-', lw=2, label='True θ₁')
ax.plot(t_ctrl, theta1_delayed, '#2E86AB', lw=2, ls='--', label='Proprioception (40ms delay)')
ax.fill_between(t_ctrl, theta1_true, theta1_delayed, color='#2E86AB', alpha=0.15)
max_stale_prop = np.max(np.abs(theta1_true - theta1_delayed))
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Shoulder angle (°)')
ax.set_title(f'A) Proprioceptive staleness\npeak: {max_stale_prop:.1f}°', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel B: Hand x-position — true vs vision (100ms delay)
ax = axes[1]
vis_delay_steps = int(0.100 / DT_CTRL)  # 20 steps
hand_x_true = hand_ilqg[:, 0] * 100  # cm
hand_x_delayed = np.zeros(N+1)
for t in range(N+1):
    t_obs = max(0, t - vis_delay_steps)
    hand_x_delayed[t] = hand_ilqg[t_obs, 0] * 100
ax.plot(t_ctrl, hand_x_true, 'k-', lw=2, label='True hand x')
ax.plot(t_ctrl, hand_x_delayed, '#E74C3C', lw=2, ls='--', label='Vision (100ms delay)')
ax.fill_between(t_ctrl, hand_x_true, hand_x_delayed, color='#E74C3C', alpha=0.15)
max_stale_vis = np.max(np.abs(hand_x_true - hand_x_delayed))
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Hand x (cm)')
ax.set_title(f'B) Visual staleness\npeak: {max_stale_vis:.1f} cm', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel C: Both staleness signals
ax = axes[2]
stale_prop = np.abs(theta1_true - theta1_delayed)
stale_vis = np.abs(hand_x_true - hand_x_delayed)
# Compute hand speed for context
hand_speed = np.sqrt(np.diff(hand_ilqg[:,0])**2 + np.diff(hand_ilqg[:,1])**2) / DT_CTRL * 100
ax2 = ax.twinx()
ax2.fill_between(t_ctrl[:-1], 0, hand_speed, color='gray', alpha=0.1, label='Hand speed')
ax.plot(t_ctrl, stale_prop, '#2E86AB', lw=2.5, label=f'Prop staleness (peak {max_stale_prop:.1f}°)')
ax.plot(t_ctrl, stale_vis, '#E74C3C', lw=2.5, label=f'Vis staleness (peak {max_stale_vis:.1f} cm)')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Staleness (° or cm)')
ax2.set_ylabel('Hand speed (cm/s)', color='gray')
ax.set_title('C) Both peak during\nmaximum velocity', fontweight='bold')
ax.legend(fontsize=8, loc='upper left'); ax.grid(True, alpha=0.3)

fig.suptitle('Figure 7 (reproduced): Sensory delays make observations stale',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print(f"Peak staleness:")
print(f"  Proprioception (40ms): {max_stale_prop:.1f}° shoulder angle")
print(f"  Vision (100ms):        {max_stale_vis:.1f} cm hand position")
print(f"\nBoth peak during maximum velocity — precisely when accurate")
print(f"state information matters most for the controller.")


### Discussion Questions

**Q8a:** The augmented state $A_{\text{aug}}$ is 336×336 but only has ${} nonzero entries. Why is the matrix so sparse? What does each identity block on the sub-diagonal represent physically?

**Q8b:** At peak velocity (~250 ms), proprioception sees the shoulder angle from 40 ms ago and vision sees the hand from 100 ms ago. If the naive KF applies these as corrections to the *current* estimate, in which direction does each correction push — toward the past or future? Why is this harmful?

**Q8c:** The augmented-state KF assigns each observation to the correct delayed slot and propagates the correction forward through the buffer. Why does this give a better estimate of the *current* state, even though the observations are still delayed?


## Summary

| Part | What you built | Lecture figure reproduced |
|------|---------------|--------------------------|
| 2 | Jacobians via finite differences | — |
| 3 | Extended Riccati with $v_t$ | — |
| 4 | Three-term forward pass | — |
| 5 | Full iLQG loop | **Figure 4** (convergence) |
| — | Jacobian analysis | **Figure 5** (B_t variation) |
| 6 | EPH vs single-pt vs iLQG | **Figure 6** (comparison) |
| 7 | 10D vs 16D state vector | — |
| 8a | Augmented-state matrices (336D) | — |
| 8b | Sensory staleness analysis | **Figure 7** (delays) |

**The takeaway:** The Riccati equation from Lab 12 works unchanged — you just feed it time-varying $A_t, B_t$ and add the $v_t$ recursion. The correct state vector (16D, not 10D) and re-linearization at every timestep are both essential. Sensory delays create significant staleness during fast movements — the augmented-state matrices from 8a are the foundation for handling these delays properly in a Kalman filter.
